# Silver Layer Multi-Table EDA

**Purpose**: This notebook provides exploratory data analysis (EDA) and data quality checks across all tables in the `databricks_bootcamp_dwb.silver` schema.

**Workflow**: 
1. Create initial drafts of silver transformation notebooks for each table
2. Use this notebook to verify data consistency across tables
3. Iterate on silver notebooks when issues are discovered

**Scope**: 
- Preview data from all silver tables
- Identify primary and foreign keys
- Test join compatibility between related tables
- Check referential integrity

**Note**: This is exploration only - no data transformations are performed here.

## 1. Identify Potential Join Columns
Identify all columns containing 'id', 'num', or 'key' as potential join columns.

In [0]:
%sql
-- Manually documented join columns based on table previews
SELECT * FROM (
  VALUES 
    ('crm_customers', 'customer_id', 'integer'),
    ('crm_customers', 'customer_key', 'string'),
    ('crm_products', 'product_id', 'integer'),
    ('crm_products', 'product_key', 'string'),
    ('crm_products', 'category_id', 'string'),
    ('crm_sales', 'order_number', 'string'),
    ('crm_sales', 'product_key', 'string'),
    ('crm_sales', 'customer_id', 'integer'),
    ('erp_customer_location', 'customer_id', 'string'),
    ('erp_customers', 'customer_id', 'string'),
    ('erp_products', 'product_id', 'string')
) AS t(table_name, column_name, data_type)
ORDER BY table_name, column_name

### Hypothesis 1: Initial Join Relationships

**crm_customers:**
* customer_id (primary key) → crm_sales.customer_id ?
* customer_id (primary key) → erp_customer_location.customer_id ?
* customer_key → erp_customers.customer_id ?

**crm_products:**
* product_id (primary key) → crm_sales.product_key
* product_key
* category_id → erp_products.product_id ?

**crm_sales:**
* order_number 
* product_key
* customer_id → erp_customer_location.customer_id ?
* customer_id → erp_customers.customer_id ?

**erp_customer_location:**
* customer_id (primary key) → erp_customers.customer_id ?

**erp_customers:**
* customer_id (primary key)

**erp_products:**
* product_id (primary key)

### 1.1 Visual Join Column Comparison
Before testing joins, display the first 100 rows of potential join columns side by side, grouped by hypothesized relationships. This visual inspection helps identify format differences and patterns.

In [0]:
%sql
-- Customer ID columns across CRM and ERP systems
SELECT 
  c.customer_id AS crm_cust_id_int,
  c.customer_key AS crm_cust_key_str,
  l.customer_id AS erp_location_cust_id,
  e.customer_id AS erp_cust_id
FROM (
  SELECT customer_id, customer_key, 
         ROW_NUMBER() OVER (ORDER BY customer_id) as rn
  FROM databricks_bootcamp_dwb.silver.crm_customers
) c
FULL OUTER JOIN (
  SELECT customer_id,
         ROW_NUMBER() OVER (ORDER BY customer_id) as rn
  FROM databricks_bootcamp_dwb.silver.erp_customer_location
) l ON c.rn = l.rn
FULL OUTER JOIN (
  SELECT customer_id,
         ROW_NUMBER() OVER (ORDER BY customer_id) as rn
  FROM databricks_bootcamp_dwb.silver.erp_customers
) e ON c.rn = e.rn
LIMIT 100

In [0]:
%sql
-- Product and category ID columns across CRM and ERP systems
SELECT 
  p.product_key AS crm_prd_key,
  p.category_id AS crm_category_id,
  e.product_id AS erp_prd_id
FROM (
  SELECT DISTINCT product_key, category_id,
         ROW_NUMBER() OVER (ORDER BY product_key) as rn
  FROM databricks_bootcamp_dwb.silver.crm_products
) p
FULL OUTER JOIN (
  SELECT DISTINCT product_id,
         ROW_NUMBER() OVER (ORDER BY product_id) as rn
  FROM databricks_bootcamp_dwb.silver.erp_products
) e ON p.rn = e.rn
LIMIT 100

In [0]:
%sql
-- Sales table join columns to CRM tables
SELECT 
  s.customer_id AS sales_cust_id,
  s.product_key AS sales_prd_key,
  s.order_number AS sales_order_num,
  c.customer_id AS crm_cust_id,
  p.product_key AS crm_prd_key
FROM (
  SELECT DISTINCT customer_id, product_key, order_number,
         ROW_NUMBER() OVER (ORDER BY customer_id) as rn
  FROM databricks_bootcamp_dwb.silver.crm_sales
) s
LEFT JOIN (
  SELECT customer_id,
         ROW_NUMBER() OVER (ORDER BY customer_id) as rn
  FROM databricks_bootcamp_dwb.silver.crm_customers
) c ON s.rn = c.rn
LEFT JOIN (
  SELECT DISTINCT product_key,
         ROW_NUMBER() OVER (ORDER BY product_key) as rn
  FROM databricks_bootcamp_dwb.silver.crm_products
) p ON s.rn = p.rn
LIMIT 100

## 2. Table Previews
Preview the first 10 rows of each table in the silver schema to understand structure and data.

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.crm_customers LIMIT 10

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.crm_products LIMIT 10

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.crm_sales LIMIT 10

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.erp_customer_location LIMIT 100

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.erp_customers LIMIT 100

In [0]:
%sql
SELECT * FROM databricks_bootcamp_dwb.silver.erp_products LIMIT 10

## 3. Primary and Foreign Key Analysis
Identify primary keys (unique identifiers) and foreign keys (relationships) in each table.

In [0]:
%sql
-- Check uniqueness of potential primary key (customer_id)
SELECT 
  'crm_customers' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customer_ids,
  CASE WHEN COUNT(*) = COUNT(DISTINCT customer_id) THEN 'YES' ELSE 'NO' END AS is_unique_key
FROM databricks_bootcamp_dwb.silver.crm_customers

In [0]:
%sql
-- Check uniqueness of potential primary key (product_id)
SELECT 
  'crm_products' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT product_id) AS distinct_product_ids,
  CASE WHEN COUNT(*) = COUNT(DISTINCT product_id) THEN 'YES' ELSE 'NO' END AS is_unique_key
FROM databricks_bootcamp_dwb.silver.crm_products

In [0]:
%sql
-- Check uniqueness of potential primary key (order_number) and identify foreign keys
SELECT 
  'crm_sales' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_number) AS distinct_order_numbers,
  CASE WHEN COUNT(*) = COUNT(DISTINCT order_number) THEN 'YES' ELSE 'NO' END AS is_unique_key,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  COUNT(DISTINCT product_key) AS distinct_products
FROM databricks_bootcamp_dwb.silver.crm_sales

In [0]:
%sql
-- Check uniqueness of potential primary key (customer_id)
SELECT 
  'erp_customer_location' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customer_ids,
  CASE WHEN COUNT(*) = COUNT(DISTINCT customer_id) THEN 'YES' ELSE 'NO' END AS is_unique_key
FROM databricks_bootcamp_dwb.silver.erp_customer_location

In [0]:
%sql
-- Check uniqueness of potential primary key (customer_id)
SELECT 
  'erp_customers' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_id) AS distinct_customer_ids,
  CASE WHEN COUNT(*) = COUNT(DISTINCT customer_id) THEN 'YES' ELSE 'NO' END AS is_unique_key
FROM databricks_bootcamp_dwb.silver.erp_customers

In [0]:
%sql
-- Check uniqueness of potential primary key (product_id)
SELECT 
  'erp_products' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT product_id) AS distinct_product_ids,
  CASE WHEN COUNT(*) = COUNT(DISTINCT product_id) THEN 'YES' ELSE 'NO' END AS is_unique_key
FROM databricks_bootcamp_dwb.silver.erp_products

## 4. Join Compatibility Testing - Wave 1
Test joins between tables that share common keys to verify referential integrity.
For failed joins, we'll display sample values to understand the mismatch.

In [0]:
%sql
-- Test join between crm_sales and crm_customers
-- Check referential integrity: are all customer_ids in sales present in customers?
SELECT 
  'crm_sales → crm_customers' AS join_test,
  COUNT(DISTINCT s.customer_id) AS sales_customers,
  COUNT(DISTINCT c.customer_id) AS matched_customers,
  COUNT(DISTINCT s.customer_id) - COUNT(DISTINCT c.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_sales s
LEFT JOIN databricks_bootcamp_dwb.silver.crm_customers c ON s.customer_id = c.customer_id

In [0]:
%sql
-- Test join between crm_sales and crm_products
-- Check referential integrity: are all product_keys in sales present in products?
SELECT 
  'crm_sales → crm_products' AS join_test,
  COUNT(DISTINCT s.product_key) AS sales_products,
  COUNT(DISTINCT p.product_key) AS matched_products,
  COUNT(DISTINCT s.product_key) - COUNT(DISTINCT p.product_key) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_sales s
LEFT JOIN databricks_bootcamp_dwb.silver.crm_products p ON s.product_key = p.product_key

In [0]:
%sql
-- Test join between erp_customer_location and erp_customers
-- Check referential integrity: are all customer_ids in location present in customers?
SELECT 
  'erp_customer_location → erp_customers' AS join_test,
  COUNT(DISTINCT l.customer_id) AS location_customers,
  COUNT(DISTINCT c.customer_id) AS matched_customers,
  COUNT(DISTINCT l.customer_id) - COUNT(DISTINCT c.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.erp_customer_location l
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customers c ON l.customer_id = c.customer_id

In [0]:
%sql
-- Failed join - Show sample values to understand the mismatch
(SELECT 'erp_customer_location' AS source, customer_id
FROM databricks_bootcamp_dwb.silver.erp_customer_location
LIMIT 3)

UNION ALL

(SELECT 'erp_customers' AS source, customer_id
FROM databricks_bootcamp_dwb.silver.erp_customers
LIMIT 3)

### Test: Can crm_customers.customer_key join to ERP systems?

In [0]:
%sql
-- Test if crm_customers.customer_key matches erp_customer_location.customer_id
SELECT 
  'crm_customers.customer_key → erp_customer_location' AS join_test,
  COUNT(DISTINCT c.customer_key) AS crm_customer_keys,
  COUNT(DISTINCT l.customer_id) AS matched_in_location,
  COUNT(DISTINCT c.customer_key) - COUNT(DISTINCT l.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_customers c
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customer_location l ON c.customer_key = l.customer_id

In [0]:
%sql
-- Test if crm_customers.customer_key matches erp_customers.customer_id
SELECT 
  'crm_customers.customer_key → erp_customers' AS join_test,
  COUNT(DISTINCT c.customer_key) AS crm_customer_keys,
  COUNT(DISTINCT e.customer_id) AS matched_in_erp,
  COUNT(DISTINCT c.customer_key) - COUNT(DISTINCT e.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_customers c
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customers e ON c.customer_key = e.customer_id

In [0]:
%sql
-- Sample values to understand the relationship
(SELECT 'crm_customers' AS source, customer_id, customer_key
FROM databricks_bootcamp_dwb.silver.crm_customers
LIMIT 3)

UNION ALL

(SELECT 'erp_customer_location' AS source, NULL as customer_id, customer_id as customer_key
FROM databricks_bootcamp_dwb.silver.erp_customer_location
LIMIT 3)

UNION ALL

(SELECT 'erp_customers' AS source, NULL as customer_id, customer_id as customer_key
FROM databricks_bootcamp_dwb.silver.erp_customers
LIMIT 3)

### Test: Can crm_products.category_id join to erp_products?

In [0]:
%sql
-- Test if crm_products.category_id matches erp_products.product_id
SELECT 
  'crm_products.category_id → erp_products' AS join_test,
  COUNT(DISTINCT p.category_id) AS crm_category_ids,
  COUNT(DISTINCT e.product_id) AS matched_in_erp,
  COUNT(DISTINCT p.category_id) - COUNT(DISTINCT e.product_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_products p
LEFT JOIN databricks_bootcamp_dwb.silver.erp_products e ON p.category_id = e.product_id

In [0]:
%sql
-- Sample values to understand the relationship
(SELECT 'crm_products' AS source, category_id, product_key, product_name
FROM databricks_bootcamp_dwb.silver.crm_products
LIMIT 3)

UNION ALL

(SELECT 'erp_products' AS source, product_id as category_id, NULL as product_key, sub_category as product_name
FROM databricks_bootcamp_dwb.silver.erp_products
LIMIT 3)

### Test: Can ERP tables join to crm_sales?

In [0]:
%sql
-- Test if crm_sales customer_ids can join to erp_customer_location
SELECT 
  'crm_sales.customer_id → erp_customer_location' AS join_test,
  COUNT(DISTINCT s.customer_id) AS sales_customers,
  COUNT(DISTINCT l.customer_id) AS matched_in_location,
  COUNT(DISTINCT s.customer_id) - COUNT(DISTINCT l.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_sales s
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customer_location l ON CAST(s.customer_id AS STRING) = l.customer_id

In [0]:
%sql
-- Test if crm_sales customer_ids can join to erp_customers
SELECT 
  'crm_sales.customer_id → erp_customers' AS join_test,
  COUNT(DISTINCT s.customer_id) AS sales_customers,
  COUNT(DISTINCT e.customer_id) AS matched_in_erp,
  COUNT(DISTINCT s.customer_id) - COUNT(DISTINCT e.customer_id) AS orphaned_records
FROM databricks_bootcamp_dwb.silver.crm_sales s
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customers e ON CAST(s.customer_id AS STRING) = e.customer_id

In [0]:
%sql
-- Test cross-system customer alignment
-- Note: CRM uses integer customer_id, ERP uses string with prefix - different key schemes
SELECT 
  (SELECT COUNT(DISTINCT customer_id) FROM databricks_bootcamp_dwb.silver.crm_customers) AS crm_customers,
  (SELECT COUNT(DISTINCT customer_id) FROM databricks_bootcamp_dwb.silver.erp_customers) AS erp_customers,
  'Different key schemes - CRM integer, ERP prefixed string' AS note

### Theory Tests: Substring Matching and Normalized Separators

**Performance Optimization Applied:**
These queries have been optimized to run faster by:
1. **Breaking into smaller steps** using CTEs (Common Table Expressions)
2. **Using INNER JOIN for matches** instead of LEFT JOIN with CASE statements
3. **Pre-computing totals separately** to avoid expensive aggregations on full joins
4. **Materializing normalized values** before joins (e.g., REPLACE operations)
5. **Sampling first** in example queries to limit the scope of LIKE operations

Original approach: Single LEFT JOIN with LIKE patterns across full tables (very slow)
Optimized approach: Separate totals → find matches with INNER JOIN → calculate statistics (much faster)

In [0]:
%sql
-- OPTIMIZED: Test if crm_customers.customer_id (integer) appears as substring in erp_customer_location.customer_id (string)
-- Step 1: Get total CRM customers
-- Step 2: Find matches using INNER JOIN (faster than LEFT JOIN with LIKE)
-- Step 3: Calculate statistics

WITH crm_totals AS (
  SELECT COUNT(DISTINCT customer_id) AS total_crm_customers
  FROM databricks_bootcamp_dwb.silver.crm_customers
),
matches AS (
  SELECT DISTINCT c.customer_id
  FROM databricks_bootcamp_dwb.silver.crm_customers c
  INNER JOIN databricks_bootcamp_dwb.silver.erp_customer_location l
    ON l.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')
)
SELECT 
  'crm_customers.customer_id substring in erp_customer_location' AS test_name,
  t.total_crm_customers AS crm_customer_ids,
  COUNT(DISTINCT m.customer_id) AS matched_as_substring,
  t.total_crm_customers - COUNT(DISTINCT m.customer_id) AS orphaned_records,
  ROUND(100.0 * COUNT(DISTINCT m.customer_id) / t.total_crm_customers, 2) AS match_percentage
FROM crm_totals t
LEFT JOIN matches m ON 1=1
GROUP BY t.total_crm_customers

In [0]:
%sql
-- OPTIMIZED: Test if crm_customers.customer_id (integer) appears as substring in erp_customers.customer_id (string)
-- Using same optimization pattern: separate totals from match finding

WITH crm_totals AS (
  SELECT COUNT(DISTINCT customer_id) AS total_crm_customers
  FROM databricks_bootcamp_dwb.silver.crm_customers
),
matches AS (
  SELECT DISTINCT c.customer_id
  FROM databricks_bootcamp_dwb.silver.crm_customers c
  INNER JOIN databricks_bootcamp_dwb.silver.erp_customers e
    ON e.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')
)
SELECT 
  'crm_customers.customer_id substring in erp_customers' AS test_name,
  t.total_crm_customers AS crm_customer_ids,
  COUNT(DISTINCT m.customer_id) AS matched_as_substring,
  t.total_crm_customers - COUNT(DISTINCT m.customer_id) AS orphaned_records,
  ROUND(100.0 * COUNT(DISTINCT m.customer_id) / t.total_crm_customers, 2) AS match_percentage
FROM crm_totals t
LEFT JOIN matches m ON 1=1
GROUP BY t.total_crm_customers

In [0]:
%sql
-- OPTIMIZED: Show examples of substring matches
-- Sample first, then join to avoid full table scans
WITH crm_sample AS (
  SELECT customer_id, customer_key
  FROM databricks_bootcamp_dwb.silver.crm_customers
  LIMIT 10
)
SELECT 
  c.customer_id AS crm_id_integer,
  c.customer_key AS crm_key,
  l.customer_id AS erp_location_id,
  e.customer_id AS erp_customers_id
FROM crm_sample c
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customer_location l 
  ON l.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')
LEFT JOIN databricks_bootcamp_dwb.silver.erp_customers e 
  ON e.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')

In [0]:
%sql
-- OPTIMIZED: Test if category_id matches product_id after normalizing separators to dash
-- Materialize the normalized ERP products first, then join

WITH crm_totals AS (
  SELECT COUNT(DISTINCT category_id) AS total_categories
  FROM databricks_bootcamp_dwb.silver.crm_products
),
erp_normalized AS (
  SELECT DISTINCT REPLACE(product_id, '_', '-') AS normalized_product_id
  FROM databricks_bootcamp_dwb.silver.erp_products
),
matches AS (
  SELECT DISTINCT p.category_id
  FROM databricks_bootcamp_dwb.silver.crm_products p
  INNER JOIN erp_normalized e
    ON e.normalized_product_id = p.category_id
)
SELECT 
  'Normalized crm_products.category_id → erp_products.product_id' AS test_name,
  t.total_categories AS crm_category_ids,
  COUNT(DISTINCT m.category_id) AS matched_after_normalization,
  t.total_categories - COUNT(DISTINCT m.category_id) AS orphaned_records,
  ROUND(100.0 * COUNT(DISTINCT m.category_id) / t.total_categories, 2) AS match_percentage
FROM crm_totals t
LEFT JOIN matches m ON 1=1
GROUP BY t.total_categories

In [0]:
%sql
-- OPTIMIZED: Show examples of normalized matches
-- Sample distinct categories first, then join
WITH crm_sample AS (
  SELECT DISTINCT category_id
  FROM databricks_bootcamp_dwb.silver.crm_products
  LIMIT 10
)
SELECT 
  p.category_id AS crm_category_original,
  e.product_id AS erp_product_original,
  REPLACE(e.product_id, '_', '-') AS erp_product_normalized,
  CASE WHEN REPLACE(e.product_id, '_', '-') = p.category_id THEN 'MATCH' ELSE 'NO MATCH' END AS match_status
FROM crm_sample p
LEFT JOIN databricks_bootcamp_dwb.silver.erp_products e 
  ON REPLACE(e.product_id, '_', '-') = p.category_id

### 🎯 Theory Test Results - VALIDATED!

#### Theory 1: Substring Matching ✅ **100% CONFIRMED**
**Finding**: `crm_customers.customer_id` (integer) appears as a substring in BOTH ERP customer ID fields:
* **erp_customer_location.customer_id**: 18,484 / 18,484 matched (100%)
* **erp_customers.customer_id**: 18,484 / 18,484 matched (100%)

**Pattern Discovered**:
* CRM customer_id: `11000` (integer)
* ERP location format: `"AW-00011000"` (contains the integer with prefix/formatting)
* ERP customers format: `"NASAW00011000"` (contains the integer with different prefix)

**Implication**: The integer customer_id from CRM is the **core identifier** embedded in all ERP customer strings. This is the true join key!

---

#### Theory 2: Normalized Separator Matching ✅ **97.3% CONFIRMED**
**Finding**: After normalizing separators (replacing `_` with `-`), product categories match almost perfectly:
* **Matched**: 36 / 37 category_ids (97.30%)
* **Orphaned**: Only 1 category_id failed to match

**Pattern Discovered**:
* CRM category_id: `"AC-BC"` (uses dash)
* ERP product_id: `"AC_BC"` (uses underscore)
* After normalization: **MATCH** ✓

**Implication**: The category codes are identical; only the delimiter differs. Normalizing separators creates a reliable join path.

In [0]:
%sql
-- OPTIMIZED: Find which category_id failed to match after normalization
-- Pre-compute normalized ERP products, then find orphans
WITH erp_normalized AS (
  SELECT DISTINCT REPLACE(product_id, '_', '-') AS normalized_product_id
  FROM databricks_bootcamp_dwb.silver.erp_products
)
SELECT 
  p.category_id AS orphaned_category,
  p.product_name,
  COUNT(*) AS product_count
FROM databricks_bootcamp_dwb.silver.crm_products p
LEFT JOIN erp_normalized e 
  ON e.normalized_product_id = p.category_id
WHERE e.normalized_product_id IS NULL
GROUP BY p.category_id, p.product_name

**Finding**: The orphaned category is `"CO-PE"` (Components - Pedals)
* Affects 7 products in CRM (all pedal-related products)
* No matching `"CO_PE"` exists in ERP products table
* This appears to be a **legitimate data gap** - the ERP system may not track pedals as a separate category
* Recommendation: Document this as a known gap; pedal sales won't have ERP category enrichment

## 5. Refined Hypothesis: Confirmed and New Relationships

Based on testing, here are the **confirmed** and **potential** join relationships:

**crm_customers:**
* customer_id (primary key) ✓
* **customer_id → erp_customer_location (substring match)** ✅ **(0% orphaned - NEW: 100% match via LIKE pattern)**
* **customer_id → erp_customers (substring match)** ✅ **(0% orphaned - NEW: 100% match via LIKE pattern)**
* customer_key → erp_customer_location.customer_id ❌ (100% orphaned - format mismatch: no dash vs dash)
* customer_key → erp_customers.customer_id ✓ (59.7% orphaned - partial match, 7,442 of 18,484 matched)

**crm_products:**
* product_id (primary key) ✓
* product_key → crm_sales.product_key ✓ (0% orphaned)
* **category_id → erp_products (normalized)** ✅ **(2.7% orphaned - NEW: 97.3% match after separator normalization)**
* category_id → erp_products.product_id ❌ (100% orphaned - delimiter mismatch without normalization)

**crm_sales:**
* order_number (NOT unique - multiple line items per order)
* product_key → crm_products.product_key ✓ (0% orphaned)
* customer_id → crm_customers.customer_id ✓ (0% orphaned)
* **customer_id → erp systems (via substring through crm_customers)** ✅ **(Bridged join now possible)**

**erp_customer_location:**
* customer_id (primary key) ✓
* customer_id → erp_customers.customer_id ❌ (100% orphaned - prefix mismatch: "AW-" vs "NASAW")

**erp_customers:**
* customer_id (primary key) ✓

**erp_products:**
* product_id (primary key) ✓

**crm_customers:**
* customer_id (primary key) ✓
* **customer_id → erp_customer_location (substring match)** ✅ **(0% orphaned - NEW: 100% match via LIKE pattern)**
* **customer_id → erp_customers (substring match)** ✅ **(0% orphaned - NEW: 100% match via LIKE pattern)**
* customer_key → erp_customer_location.customer_id ❌ (100% orphaned - format mismatch: no dash vs dash)
* customer_key → erp_customers.customer_id ✓ (59.7% orphaned - partial match, 7,442 of 18,484 matched)

**crm_products:**
* product_id (primary key) ✓
* product_key → crm_sales.product_key ✓ (0% orphaned)
* **category_id → erp_products (normalized)** ✅ **(2.7% orphaned - NEW: 97.3% match after separator normalization)**
* category_id → erp_products.product_id ❌ (100% orphaned - delimiter mismatch without normalization)

**crm_sales:**
* order_number (NOT unique - multiple line items per order)
* product_key → crm_products.product_key ✓ (0% orphaned)
* customer_id → crm_customers.customer_id ✓ (0% orphaned)
* **customer_id → erp systems (via substring through crm_customers)** ✅ **(Bridged join now possible)**

**erp_customer_location:**
* customer_id (primary key) ✓
* customer_id → erp_customers.customer_id ❌ (100% orphaned - prefix mismatch: "AW-" vs "NASAW")

**erp_customers:**
* customer_id (primary key) ✓

**erp_products:**
* product_id (primary key) ✓

In [0]:
%sql
-- Test cross-system product alignment
-- Note: CRM uses numeric product_id, ERP uses string product_id - different key schemes
SELECT 
  (SELECT COUNT(DISTINCT product_id) FROM databricks_bootcamp_dwb.silver.crm_products) AS crm_products,
  (SELECT COUNT(DISTINCT product_id) FROM databricks_bootcamp_dwb.silver.erp_products) AS erp_products,
  'Different key schemes - CRM numeric, ERP categorical' AS note

## 6. Summary and Next Steps

### 🎉 MAJOR BREAKTHROUGH - Substring Pattern Discovered!

### Perfect Matches ✅ (0% orphaned):

**Within CRM System:**
* **crm_sales → crm_customers** (via customer_id)
* **crm_sales → crm_products** (via product_key)

**CRM to ERP - Customer Bridge (NEW!):**
* **crm_customers.customer_id → erp_customer_location** (100% match via substring/LIKE)
  * Pattern: Integer `11000` found in `"AW-00011000"`
  * Join strategy: `WHERE erp.customer_id LIKE CONCAT('%', CAST(crm.customer_id AS STRING), '%')`
* **crm_customers.customer_id → erp_customers** (100% match via substring/LIKE)
  * Pattern: Integer `11000` found in `"NASAW00011000"`
  * Same join strategy as above

**CRM to ERP - Product Bridge (NEW!):**
* **crm_products.category_id → erp_products.product_id** (97.3% match after normalization)
  * Only 1 out of 37 categories failed to match
  * Join strategy: `WHERE REPLACE(erp.product_id, '_', '-') = crm.category_id`

---

### Partial Match - Still Relevant (59.7% orphaned):
* **crm_customers.customer_key → erp_customers**: 7,442 of 18,484 customers matched
  * Alternative bridge, but substring method above is superior (100% coverage)

---

### Failed Joins - Less Critical Now ❌:

**Within ERP System:**
* **erp_customer_location → erp_customers** (100% orphaned)
  * Issue: Different prefix formats (`"AW-"` vs `"NASAW"`)
  * Not critical for CRM-to-ERP joins since both can now join independently to CRM

---

### 🛠️ Recommended Join Patterns for Gold Layer:

#### Customer Enrichment (Sales → CRM → ERP):
```sql
-- Full customer journey with location and demographics
SELECT 
  s.*,
  c.customer_key,
  l.city, l.state_province, l.country_region,
  e.birth_date, e.marital_status, e.yearly_income
FROM crm_sales s
JOIN crm_customers c ON s.customer_id = c.customer_id
LEFT JOIN erp_customer_location l 
  ON l.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')
LEFT JOIN erp_customers e 
  ON e.customer_id LIKE CONCAT('%', CAST(c.customer_id AS STRING), '%')
```

#### Product Category Enrichment:
```sql
-- Link CRM products to ERP product categories
SELECT 
  p.*,
  e.sub_category,
  e.product_id as erp_category_code
FROM crm_products p
LEFT JOIN erp_products e 
  ON REPLACE(e.product_id, '_', '-') = p.category_id
```

---

### Action Items:
1. **✅ VALIDATED**: Substring join pattern works for 100% of customer records
2. **✅ VALIDATED**: Normalized separator join works for 97.3% of product categories
3. **Next**: Investigate the 1 orphaned product category (2.7%)
4. **Next**: Implement these join patterns in gold layer transformations
5. **Next**: Document substring and normalization patterns as standard join conventions
6. **Optional**: Still standardize string formats in silver for cleaner direct equality joins